# Stage 1 — Layer Stability

**Objective:** For each of the four candidate multicircuit groups, compute intra-group mean Jaccard at every layer (0–7). Does the grouping hold across the network, or is it confined to one layer?

**Pass criterion:** The group shows significantly elevated intra-group similarity at 3 or more layers.

See `multicircuits.md` for full definitions and experiment plan.

In [ ]:
# Cell 1 – Setup & load atlas
import subprocess, sys, os, shutil
for pkg in ["h5py", "seaborn", "matplotlib", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)

DATASET = "115_strong"

_DATASETS = {
    "115_strong": "universal_115x100_strong",
    "small":      "atlas_small_40x50x50_validated_prompts",
}
_base = _DATASETS[DATASET]

if IN_COLAB:
    DATA_DIR = "/content/drive/MyDrive/DATA/CSP-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/CSP-Atlas"

ATLAS_HDF5 = f"{DATA_DIR}/{_base}.h5"

LOCAL_SRC = "/Users/piotrwilam/Code/CSP-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/CSP-Atlas/src"
SRC_PATH  = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from module2.io_utils import load_atlas_hdf5

print(f"Environment : {'Colab' if IN_COLAB else 'Local'}")
print(f"Loading     : {ATLAS_HDF5}")
atlas = load_atlas_hdf5(ATLAS_HDF5)
pair_masks      = atlas["pair_masks"]
universal_masks = atlas["universal_masks"]
metadata        = atlas["metadata"]

print(f"Pairs       : {len(pair_masks)}")
print(f"AST nodes   : {len(universal_masks['ast'])}")
print(f"Builtins    : {len(universal_masks['builtin'])}")

In [ ]:
# Cell 2 – Build per-layer mask matrices & define candidate groups
layer_ids = sorted({lid for lm in pair_masks.values() for lid in lm})

# Build circuit name list and full mask matrix
circuit_names = []
mask_rows = []
for name in sorted(universal_masks["ast"]):
    vec = []
    for lid in layer_ids:
        m = universal_masks["ast"][name].get(lid)
        if m is not None:
            vec.append(m.astype(np.float32))
    if vec:
        circuit_names.append(f"AST:{name}")
        mask_rows.append(np.concatenate(vec))

for name in sorted(universal_masks["builtin"]):
    vec = []
    for lid in layer_ids:
        m = universal_masks["builtin"][name].get(lid)
        if m is not None:
            vec.append(m.astype(np.float32))
    if vec:
        circuit_names.append(f"BLT:{name}")
        mask_rows.append(np.concatenate(vec))

mask_matrix = np.array(mask_rows)
n_circuits, n_neurons = mask_matrix.shape
neurons_per_layer = n_neurons // len(layer_ids)

# ── Candidate multicircuit groups ────────────────────────────────────────
GROUPS = {
    "A: Generic AST": {
        "AST:AsyncWith", "AST:Attribute", "AST:ClassDef", "AST:Delete",
        "AST:Dict", "AST:GeneratorExp", "AST:Lambda", "AST:Return",
        "AST:SetComp", "AST:Slice", "AST:Subscript", "AST:YieldFrom",
        "BLT:classmethod", "BLT:isinstance", "BLT:property", "BLT:repr",
        "BLT:staticmethod", "BLT:super", "BLT:zip",
    },
    "B: Exceptions": {
        "AST:Continue", "AST:DictComp", "AST:ListComp", "AST:With", "AST:Yield",
        "BLT:AttributeError", "BLT:Exception", "BLT:FileNotFoundError",
        "BLT:IndexError", "BLT:KeyError", "BLT:MemoryError", "BLT:NameError",
        "BLT:OSError", "BLT:OverflowError", "BLT:RecursionError",
        "BLT:RuntimeError", "BLT:StopIteration", "BLT:TypeError",
        "BLT:ValueError", "BLT:ZeroDivisionError",
    },
    "C: Raise+Errors": {
        "AST:While", "AST:Raise", "BLT:ArithmeticError",
        "BLT:ImportError", "BLT:LookupError", "BLT:NotImplementedError",
    },
    "D: Control flow": {
        "AST:For", "AST:AsyncFor", "AST:Global", "AST:ImportFrom",
        "AST:Nonlocal", "AST:Starred", "AST:ExceptHandler", "AST:Try",
        "AST:If", "AST:IfExp", "AST:FunctionDef", "AST:AsyncFunctionDef",
        "AST:Break",
    },
}

# Map names to indices
group_indices = {}
for gname, members in GROUPS.items():
    idx = [i for i, c in enumerate(circuit_names) if c in members]
    found = {circuit_names[i] for i in idx}
    missing = members - found
    if missing:
        print(f"WARNING: {gname} — missing: {missing}")
    group_indices[gname] = idx
    print(f"{gname}: {len(idx)} circuits")

print(f"\nLayers: {layer_ids} | Neurons/layer: {neurons_per_layer}")

In [ ]:
# Cell 3 – Compute intra-group Jaccard at every layer
import numpy as np

def jaccard_at_layer(mask_matrix, layer_idx, neurons_per_layer, indices):
    """Compute mean pairwise Jaccard for a group at one layer (selective neurons only)."""
    s = layer_idx * neurons_per_layer
    e = s + neurons_per_layer
    layer_slice = mask_matrix[:, s:e]

    # Selective neurons at this layer
    col_sums = layer_slice.sum(axis=0)
    n_total = len(col_sums)
    selective = (col_sums > 0) & (col_sums < mask_matrix.shape[0])
    layer_sel = layer_slice[:, selective]

    if layer_sel.shape[1] == 0:
        return 0.0, 0, int(selective.sum())

    vals = []
    for ii in range(len(indices)):
        a = layer_sel[indices[ii]].astype(bool)
        for jj in range(ii + 1, len(indices)):
            b = layer_sel[indices[jj]].astype(bool)
            inter = (a & b).sum()
            union = (a | b).sum()
            vals.append(inter / union if union > 0 else 0.0)

    return np.mean(vals) if vals else 0.0, len(vals), int(selective.sum())


# Compute for all groups x all layers
results = []
for gname, indices in group_indices.items():
    for li, lid in enumerate(layer_ids):
        mean_jac, n_pairs, n_selective = jaccard_at_layer(
            mask_matrix, li, neurons_per_layer, indices)
        results.append({
            "group": gname, "layer": lid,
            "mean_jaccard": mean_jac, "n_pairs": n_pairs,
            "n_selective": n_selective, "group_size": len(indices),
        })

results_df = pd.DataFrame(results)

# Pivot for display
pivot = results_df.pivot(index="group", columns="layer", values="mean_jaccard")
print("Mean intra-group Jaccard per layer:")
print(pivot.round(4).to_string())

# Count layers with elevated similarity (> 0.1 as a baseline)
print("\n\nLayers with mean Jaccard > 0.1:")
for gname in GROUPS:
    g_data = results_df[results_df["group"] == gname]
    elevated = g_data[g_data["mean_jaccard"] > 0.1]
    layers = elevated["layer"].tolist()
    print(f"  {gname}: {len(layers)} layers — {layers}")

In [ ]:
# Cell 4 – Layer profile plot
import matplotlib.pyplot as plt

colors = {"A: Generic AST": "#2c7bb6", "B: Exceptions": "#d7191c",
          "C: Raise+Errors": "#fdae61", "D: Control flow": "#abd9e9"}

fig, ax = plt.subplots(figsize=(10, 5))

for gname, indices in group_indices.items():
    g_data = results_df[results_df["group"] == gname]
    ax.plot(g_data["layer"], g_data["mean_jaccard"],
            marker="o", linewidth=2, label=gname, color=colors[gname])

ax.set_xlabel("Layer")
ax.set_ylabel("Mean intra-group Jaccard")
ax.set_title("Stage 1: Layer Stability — Intra-group similarity across layers")
ax.set_xticks(layer_ids)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 5 – Per-group heatmaps at each layer (Jaccard within group)
import seaborn as sns, matplotlib.pyplot as plt, matplotlib.colors as mcolors

_colors_5 = ["#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"]
_cmap_5   = mcolors.ListedColormap(_colors_5)
_bounds_5 = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
_norm_5   = mcolors.BoundaryNorm(_bounds_5, _cmap_5.N)

for gname, indices in group_indices.items():
    g_names = [circuit_names[i] for i in indices]
    n_g = len(indices)

    # Pick layers with any selective neurons
    active_layers = []
    for li, lid in enumerate(layer_ids):
        row = results_df[(results_df["group"] == gname) & (results_df["layer"] == lid)]
        if row["n_selective"].values[0] > 0:
            active_layers.append((li, lid))

    if not active_layers:
        print(f"\n{gname}: no selective neurons at any layer — skipping")
        continue

    n_cols = min(4, len(active_layers))
    n_rows = (len(active_layers) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(5 * n_cols, max(4, n_g * 0.2 + 1) * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes[np.newaxis, :]
    elif n_cols == 1:
        axes = axes[:, np.newaxis]

    for plot_idx, (li, lid) in enumerate(active_layers):
        r, c = divmod(plot_idx, n_cols)
        ax = axes[r, c]

        s = li * neurons_per_layer
        e = s + neurons_per_layer
        layer_slice = mask_matrix[:, s:e]
        col_s = layer_slice.sum(axis=0)
        selective = (col_s > 0) & (col_s < n_circuits)
        layer_sel = layer_slice[:, selective]

        # Pairwise Jaccard
        jmat = np.zeros((n_g, n_g))
        for ii in range(n_g):
            a = layer_sel[indices[ii]].astype(bool)
            for jj in range(ii, n_g):
                b = layer_sel[indices[jj]].astype(bool)
                inter = (a & b).sum()
                union = (a | b).sum()
                jac = inter / union if union > 0 else 0.0
                jmat[ii, jj] = jmat[jj, ii] = jac

        sns.heatmap(jmat, ax=ax, vmin=0, vmax=1, cmap=_cmap_5, norm=_norm_5,
                    xticklabels=g_names, yticklabels=g_names,
                    cbar=False)
        ax.set_title(f"Layer {lid} ({int(selective.sum())} sel.)")
        ax.tick_params(labelsize=5)

    # Hide unused subplots
    for plot_idx in range(len(active_layers), n_rows * n_cols):
        r, c = divmod(plot_idx, n_cols)
        axes[r, c].set_visible(False)

    fig.suptitle(f"{gname} — intra-group Jaccard per layer", fontsize=13)
    plt.tight_layout(); plt.show()

In [ ]:
# Cell 6 – Selective neuron context: how many selective neurons exist at each layer
sel_per_layer = []
for li, lid in enumerate(layer_ids):
    s = li * neurons_per_layer
    e = s + neurons_per_layer
    layer_slice = mask_matrix[:, s:e]
    col_s = layer_slice.sum(axis=0)
    n_sel = int(((col_s > 0) & (col_s < n_circuits)).sum())
    sel_per_layer.append(n_sel)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(layer_ids, sel_per_layer, color="#2c7bb6", edgecolor="black")
for lid, ns in zip(layer_ids, sel_per_layer):
    ax.text(lid, ns + 0.5, str(ns), ha="center", fontsize=9)
ax.set_xlabel("Layer")
ax.set_ylabel("# selective neurons")
ax.set_title("Selective neurons per layer (context for Jaccard values)")
ax.set_xticks(layer_ids)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 7 – Verdict
THRESHOLD = 0.1  # minimum mean Jaccard to count as "elevated"
MIN_LAYERS = 3   # must pass at this many layers

print("=" * 65)
print("STAGE 1 VERDICT — LAYER STABILITY")
print("=" * 65)

for gname in GROUPS:
    g_data = results_df[results_df["group"] == gname]
    elevated = g_data[g_data["mean_jaccard"] > THRESHOLD]
    n_elevated = len(elevated)
    layers = elevated["layer"].tolist()
    best_layer = g_data.loc[g_data["mean_jaccard"].idxmax()]

    passed = n_elevated >= MIN_LAYERS
    status = "PASS" if passed else "FAIL"

    print(f"\n{gname}")
    print(f"  Elevated layers (J > {THRESHOLD}): {n_elevated} — {layers}")
    print(f"  Best layer: {int(best_layer['layer'])} (J = {best_layer['mean_jaccard']:.4f})")
    print(f"  Verdict: {status}")

print("\n" + "=" * 65)
survivors = [g for g in GROUPS
             if len(results_df[(results_df["group"] == g) &
                               (results_df["mean_jaccard"] > THRESHOLD)]) >= MIN_LAYERS]
print(f"\nSurvivors for Stage 2: {survivors if survivors else 'NONE'}")